# Classical Imports .......

In [1]:
import numpy as np 
import matplotlib.pyplot as plt 
from glob import glob 
import os 
import random 
import pickle
import torch 
import torch.nn as nn
from torch.nn import functional as F
import sys



# Set Random Seed for reproducibility

In [2]:

torch_seed = 42
torch.manual_seed(torch_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(torch_seed)


random_seed = 42
random.seed(random_seed)

np_seed = 42
np.random.seed(np_seed)


# Set device for computation 

In [3]:

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


# Set Hyper parameters ......

In [13]:
batch_size = 128
block_size = 128

max_iters = 10000
eval_iters = 10

learning_rate = 1e-4
n_embd = 256
n_head = 8
n_layer = 8
dropout = 0.2

# Load Vocabulary .....

In [4]:

vocab = np.loadtxt("vocab.txt")
vocab_size = len(vocab)
print(vocab_size)

3


# Load data for training .....

In [5]:
def get_random_chunk(split):
    filename = "train_data.txt" if split == 'train' else "val_data.txt"
    file = np.loadtxt(filename)
    file_size = len(file)
    start_pos = random.randint(0, (file_size) - block_size*batch_size)
    # print("start_pos = ", start_pos)
    chunk = np.loadtxt(filename, skiprows=start_pos, max_rows=(batch_size * block_size)-1)
    data = torch.tensor(chunk, dtype=torch.long)
    return data
    
def get_batch(split):
    data = get_random_chunk(split)
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


# Define the loss function for training .....

In [6]:

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


# Lets define the model ......

In [ ]:
# Attention block

In [1]:

class Attention_Block(nn.Module):

    def __init__(self, head_size, n_embd, block_size, dropout=0.2):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input-->(batch, time_step, vocab)
        B,T,C = x.shape
        k = self.key(x)   
        q = self.query(x) 
        attention = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 
        causal_attention = attention.masked_fill(self.tril[:T, :T] == 0, float('-inf')) 
        weighted_Attn= F.softmax(causal_attention, dim=-1) 
        weighted_Attn = self.dropout(weighted_Attn)
        v = self.value(x) 
        context_matrix = weighted_Attn @ v 
        return context_matrix
        

NameError: name 'nn' is not defined

# Multi Head Attention

In [23]:

class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size, n_embd, block_size, dropout = 0.2):
        super().__init__()
        self.heads = nn.ModuleList([Attention_Block(head_size, n_embd, block_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1) 
        out = self.dropout(self.proj(out))
        return out

# Feed Forward Layer

In [24]:
class FeedFoward(nn.Module):

    def __init__(self, n_embd, dropout = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

# Lets assemble the Trandformer blocks .....

In [25]:
class Transformer_Block(nn.Module):

    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size, n_embd, block_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        y = self.sa(x)
        x = self.ln1(x + y)
        y = self.ffwd(x)
        x = self.ln2(x + y)
        return x

In [43]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd, block_size, n_layer, n_head, device):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Transformer_Block(n_embd, n_head=n_head, block_size = block_size) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) 
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.device = device
        
        
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        # print(index.shape)
        B, T = index.shape
        tok_emb = self.token_embedding_table(index) # (B,T,V), B--->Batch T--->Time step, V--->vocab size
        #pos_emb = self.position_embedding_table(torch.arange(T, device=self.device)) 
        pos_emb = self.position_embedding_table(torch.arange(T, device=index.device))
        x = tok_emb + pos_emb # (B,T,V)
        x = self.blocks(x) # (B,T,V)
        x = self.ln_f(x) # (B,T,V)
        logits = self.lm_head(x) # (B,T,V)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            # print(logits.shape, targets)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, index, block_size, max_new_tokens):
        for _ in range(max_new_tokens):
            # croping
            index_cond = index[:, -block_size:]
            logits, loss = self.forward(index_cond)
            # last time step
            logits = logits[:, -1, :] #(B, V)
            probs = F.softmax(logits, dim=-1) # (B, V)
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index

# Lets Initialize the Model

In [46]:

model = GPTLanguageModel(vocab_size, n_embd, block_size, n_layer, n_head, device).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Training ......

In [28]:

train_loss = []
val_loss = []
epochs = []
best_loss = float('inf')

for iter in range(max_iters):
    # print(iter)
    if iter % eval_iters == 0:
        losses = estimate_loss()
        train_loss.append(losses['train'].item())
        val_loss.append(losses['val'].item())
        epochs.append(iter)
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}",flush=True)
        if(losses['val'].item()<best_loss):
            best_loss = losses['val'].item()
            with open(f'model_best_three_well.pkl', 'wb') as f:
                pickle.dump(model, f)

    xb, yb = get_batch('train')

    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()



with open(f'model_three_well.pkl', 'wb') as f:
    pickle.dump(model, f)
print('model saved')


train_loss = np.array(train_loss)
val_loss = np.array(val_loss)
epochs = np.array(epochs)
np.savetxt("train_val_loss_three_well.txt", np.array([epochs, train_loss, val_loss]).T, fmt = "%0.3e", delimiter = "\t")


step: 0, train loss: 1.083, val loss: 1.093


KeyboardInterrupt: 

# Lets generate from saved model ......

In [37]:
import torch
import pickle
import io

class CPU_Unpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == 'torch.storage' and name == '_load_from_bytes':
            return lambda b: torch.load(
                io.BytesIO(b),   
                map_location='cpu',
                weights_only=False
            )
        return super().find_class(module, name)

print("loading model...")

with open("model_best_3state_latent_2d_run_1.pkl", "rb") as f:
    model = CPU_Unpickler(f).load()

print("loaded successfully!")

loading model...
loaded successfully!


In [47]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
prompt = np.loadtxt("prompt.txt")
prompt = prompt[:128]
context = torch.tensor(prompt, dtype=torch.long, device=device)
generated_chars = model.generate(context.unsqueeze(0), block_size = block_size, max_new_tokens=20000)[0].tolist()
np.savetxt(f"generated_state.txt", np.array([generated_chars]).T, fmt = "%d")

cpu
